# Session 1: a light switch that watches the dark
**A. Real-world problem.** A garden path light turns itself on at dusk and off at dawn, with no
one touching a switch. It needs a *sensor* (something that feels light), a *decision* (is it dark
enough?) and a *switch* that powers the lamp.

**Plan.** (1) the light sensor (an LDR) and how its resistance changes, (2) the decision,
(3) the block diagram of the whole controller, (4) plot both.

The numbers here come from an *assumed, illustrative* LDR curve, not from a real data sheet.

In [ ]:
import pathlib

import numpy as np

OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"
R10 = 10_000.0     # ohm: assumed LDR resistance at 10 lux (illustrative)
GAMMA = 0.7        # assumed slope of the LDR curve on a log-log graph (illustrative)

## Step 1: the LDR
LDR = light dependent resistor. In bright light its resistance is small; in the dark it is large.
Light is measured in *lux* (a candle-lit room is a few lux, an office about 300 lux, a sunny
day tens of thousands). Our illustrative model:

    R_ldr = R10 * (lux / 10) ** (-GAMMA)

Check by hand: at 10 lux the bracket is 1, so R = 10 000 ohm. At 100 lux the bracket is 10 and
10 ** (-0.7) = 0.2, so R is about 2 000 ohm: ten times more light, five times less resistance.

In [ ]:
def ldr_resistance(lux, r10=R10, gamma=GAMMA):
    """LDR resistance in ohm for a light level in lux (lux must be above 0)."""
    lux = np.asarray(lux, dtype=float)
    return r10 * (lux / 10.0) ** (-gamma)

## Step 2: the decision
The controller compares the light with a *threshold*. Below it, the LED is on.

In [ ]:
def led_is_on(lux, threshold_lux):
    """True when it is darker than the threshold (strictly below it)."""
    return np.asarray(lux, dtype=float) < threshold_lux

## Step 3: the block diagram
Four blocks in a chain; the signal flows left to right:

    LIGHT -> [ LDR + divider ] -> [ comparator ] -> [ transistor switch ] -> [ LED ]
             "how bright?"       "dark enough?"      "power the LED"

Sessions 2 and 3 turn each block into numbers.

In [ ]:
BLOCKS = ["LDR and divider (senses light)", "comparator (decides: dark enough?)",
          "transistor switch (carries the LED current)", "LED and resistor (the lamp)"]


def describe_chain(lux, threshold_lux):
    """One line per block for a given light level, so you can follow the signal."""
    on = bool(led_is_on(lux, threshold_lux))
    r = float(ldr_resistance(lux))
    return [f"light = {lux:g} lux",
            f"LDR resistance = {r:,.0f} ohm",
            f"{lux:g} lux is {'below' if on else 'above'} the {threshold_lux:g} lux threshold",
            f"LED is {'ON' if on else 'OFF'}"]

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    for lux in (1, 10, 100, 1000):
        print(f"{lux:5d} lux -> LDR {float(ldr_resistance(lux)):10,.0f} ohm")
    for line in describe_chain(5, 20):
        print(line)
    lux = np.logspace(0, 3, 200)
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.loglog(lux, ldr_resistance(lux), color="tab:blue")
    ax.plot([10], [R10], "ko")
    ax.annotate("10 lux: 10 000 ohm", (10, R10), textcoords="offset points", xytext=(8, 8))
    ax.set_xlabel("light level (lux)")
    ax.set_ylabel("LDR resistance (ohm)")
    ax.set_title("Illustrative LDR: resistance falls as light rises")
    ax.grid(alpha=0.3, which="both")
    fig.tight_layout()
    fig.savefig(OUT / "ldr_curve.png", dpi=120)

    fig2, ax2 = plt.subplots(figsize=(6, 3.2))
    ax2.semilogx(lux, led_is_on(lux, 20).astype(float), drawstyle="steps-post", color="tab:orange")
    ax2.axvline(20, color="gray", ls=":")
    ax2.set_yticks([0, 1])
    ax2.set_yticklabels(["OFF", "ON"])
    ax2.set_xlabel("light level (lux)")
    ax2.set_ylabel("LED state")
    ax2.set_title("Threshold 20 lux: LED on only below it")
    ax2.grid(alpha=0.3)
    fig2.tight_layout()
    fig2.savefig(OUT / "led_state.png", dpi=120)
    print("saved figures in", OUT)

## Try it
Do the coding tasks, answer the four questions in the README (section F), then run the checkpoint.

## Coding tasks
1. **Easy.** Change the threshold in `lesson.py` to 50 lux and print the LED state for 5, 30 and 100 lux. Predict first.
2. **Stretch.** Write `resistance_ratio(lux1, lux2)` that returns R(lux1) / R(lux2) and check it equals (lux2 / lux1) ** 0.7.
3. **Explain.** Explain, in a sentence, why the LDR curve is a straight line on a log-log graph.

## Assessment
- **A. Conceptual.** Why is a sensor whose resistance changes useful for a controller? Why do we also need a separate 'decision' block instead of connecting the LDR straight to the LED?
- **B. Numerical.** Using R = 10 000 * (lux / 10) ^ (-0.7), find the LDR resistance at 1 lux and at 1 000 lux. By what factor do they differ? Sanity check the factor.
- **C. Graph and simulation interpretation.** In the app's graph, the curve is steep on the left and nearly flat on the right. What does that say about how well an LDR distinguishes 5 lux from 15 lux compared with 405 lux from 415 lux?
- **D. Design and optimisation challenge.** Design: a path light should turn on at dusk. Compare thresholds of 5 lux and 50 lux. Which would you choose and why? Give the LDR resistance at each threshold.

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    checks.append(("10 lux gives 10 000 ohm (the model's anchor)", abs(float(L.ldr_resistance(10.0)) - 10_000.0) < 1e-6))
    r100 = float(L.ldr_resistance(100.0))
    checks.append((f"worked example: 100 lux gives about 1 995 ohm (got {r100:.0f})", abs(r100 - 1995.3) < 1.0))
    r2 = float(L.ldr_resistance(2.0))
    checks.append((f"worked example: 2 lux gives about 30 850 ohm (got {r2:.0f})", abs(r2 - 10_000 * 0.2 ** -0.7) < 1.0 and abs(r2 - 30_852) < 5))
    lux = np.logspace(-1, 4, 60)
    checks.append(("resistance falls steadily as light rises", bool(np.all(np.diff(L.ldr_resistance(lux)) < 0))))
    checks.append(("ten times more light divides resistance by 10 ** 0.7 = 5.01",
                   abs(float(L.ldr_resistance(10.0) / L.ldr_resistance(100.0)) - 10 ** 0.7) < 1e-9))
    checks.append(("dark (5 lux) switches the LED on at a 20 lux threshold", bool(L.led_is_on(5.0, 20.0))))
    checks.append(("bright (300 lux) keeps the LED off", not bool(L.led_is_on(300.0, 20.0))))
    checks.append(("exactly at the threshold the LED stays off (strict comparison)", not bool(L.led_is_on(20.0, 20.0))))
    checks.append(("the block diagram has four blocks", len(L.BLOCKS) == 4))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
